In [1]:
# ------------------------------------------------------------------------------
# FASE 0: CONFIGURACIÓN DEL ENTORNO Y UTILIDADES DE NORMALIZACIÓN
# ------------------------------------------------------------------------------


import pandas as pd
import numpy as np
import os
import unicodedata
import re
from functools import reduce

# 1. CONFIGURACIÓN DE RUTAS
BASE_PATH = r"C:\Users\Free2\Desktop\Proyecto_Localia_Rendimiento\Datos"
PATHS = {
    "partidos": os.path.join(BASE_PATH, "1_Partidos"),
    "hinchada": os.path.join(BASE_PATH, "2_Hinchada"),
    "estadios": os.path.join(BASE_PATH, "3_Estadios"),
    "finanzas": os.path.join(BASE_PATH, "4_Finanzas")
}

# 2. DICCIONARIO MAESTRO DE MAPEADO (Normalización de Identidades)
MAPPING_EQUIPOS = {
    'Aldosivi': 'Aldosivi', 'Arg Juniors': 'Argentinos Juniors', 'Argentinos': 'Argentinos Juniors',
    'Argentinos Jrs': 'Argentinos Juniors', 'Arsenal': 'Arsenal de Sarandi', 'Arsenal de Sarandí': 'Arsenal de Sarandi',
    'Atle Tucuman': 'Atletico Tucuman', 'Atle Tucuman': 'Atletico Tucuman', 'Atletico Tucuman': 'Atletico Tucuman',
    'Atlético Tucumán': 'Atletico Tucuman', 'Atl Rafaela': 'Atletico de Rafaela', 'Atlético Rafaela': 'Atletico de Rafaela',
    'Banfield': 'Banfield', 'Barracas Central': 'Barracas Central', 'Belgrano': 'Belgrano',
    'Boca Juniors': 'Boca Juniors', 'Central Cba (SdE)': 'Central Cordoba', 'Central Cordoba-SdE': 'Central Cordoba',
    'Cen. Cordoba-SdE': 'Central Cordoba', 'Chacarita': 'Chacarita Juniors', 'Chacarita': 'Chacarita Juniors',
    'Colon': 'Colon', 'Def y Justicia': 'Defensa y Justicia', 'Defensa y Justicia': 'Defensa y Justicia',
    'Deportivo Riestra': 'Deportivo Riestra', 'Estudiantes (LP)': 'Estudiantes de La Plata',
    'Estudiantes-LP': 'Estudiantes de La Plata', 'Gimnasia (LP)': 'Gimnasia y Esgrima La Plata',
    'Gimnasia-LP': 'Gimnasia y Esgrima La Plata', 'Godoy Cruz': 'Godoy Cruz', 'Huracan': 'Huracan',
    'Independiente': 'Independiente', 'Ind. Rivadavia': 'Independiente Rivadavia', 'Instituto': 'Instituto',
    'Lanus': 'Lanus', "Newell's": "Newell's Old Boys", "Newell's Old Boys": "Newell's Old Boys",
    'Patronato': 'Patronato', 'Platense': 'Platense', 'Quilmes': 'Quilmes', 'Racing Club': 'Racing Club',
    'River Plate': 'River Plate', 'Rosario Central': 'Rosario Central', 'San Lorenzo': 'San Lorenzo de Almagro',
    'San Martin (SJ)': 'San Martin (San Juan)', 'San Martin SJ': 'San Martin (San Juan)',
    'San Martin (T)': 'San Martin (Tucuman)', 'San Martin': 'San Martin (Tucuman)',
    'Sarmiento (J)': 'Sarmiento', 'Sarmiento-J': 'Sarmiento', 'Sarmiento': 'Sarmiento',
    'Talleres (C)': 'Talleres', 'Talleres-C': 'Talleres', 'Talleres': 'Talleres',
    'Tigre': 'Tigre', 'Union': 'Union', 'Velez': 'Velez Sarsfield', 'Velez Sarsfield': 'Velez Sarsfield',
    'Temperley': 'Temperley', 'Olimpo': 'Olimpo', 'Crucero (M)': 'Crucero del Norte'
}

# 3. FUNCIONES AUXILIARES
def normalizar_texto(texto):
    if not isinstance(texto, str): return texto
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    return texto.strip()

def aplicar_mapeo(df, columnas):
    for col in columnas:
        if col in df.columns:
            df[col] = df[col].str.strip().replace(MAPPING_EQUIPOS).apply(normalizar_texto)
    return df

print("✅ Entorno de estandarización configurado.")

✅ Entorno de estandarización configurado.


In [2]:
# ------------------------------------------------------------------------------
# FASE 1: ESTANDARIZACIÓN DE RESULTADOS Y VALORES DE MERCADO (FBref & AFA)
# ------------------------------------------------------------------------------


# 1. Procesamiento de Dataset FBref
df_fbref = pd.read_csv(os.path.join(PATHS["partidos"], "dataset_modelo_localia.csv"))
df_fbref = aplicar_mapeo(df_fbref, ['Home', 'Away'])

# 2. Procesamiento de Dataset AFA (2015-2022)
df_afa = pd.read_csv(os.path.join(PATHS["partidos"], "afa_2015_2022_spa.csv"))
cols_drop = ['altura_media_local', 'altura_media_visitante', 'edad_media_local', 'edad_media_visitante',
             'proporcion_zurdos_local', 'proporcion_zurdos_visitante', 'fecha_encuentro', 'fecha',
             'apuesta_local', 'apuesta_visitante', 'apuesta_empate']
df_afa = df_afa.drop(columns=cols_drop, errors='ignore')
df_afa = aplicar_mapeo(df_afa, ['equipo_local', 'equipo_visitante'])

# 3. Normalización de columnas financieras en AFA
df_afa = df_afa.rename(columns={
    'valor_mercado_local': 'Valor_Local_USD_M',
    'valor_mercado_visitante': 'Valor_Visitante_USD_M'
})

# Exportación
df_fbref.to_csv(os.path.join(PATHS["partidos"], "dataset_modelo_localia_estandarizado.csv"), index=False, encoding='utf-8-sig')
df_afa.to_csv(os.path.join(PATHS["partidos"], "afa_2015_2022_estandarizado.csv"), index=False, encoding='utf-8-sig')

print("✅ Partidos estandarizados correctamente.")

✅ Partidos estandarizados correctamente.


In [3]:
# ------------------------------------------------------------------------------
# FASE 2: PROCESAMIENTO DE ASISTENCIA, CENSOS Y POPULARIDAD
# ------------------------------------------------------------------------------


# 1. Estandarización de Asistencia (Transfermarkt)
df_asis = pd.read_csv(os.path.join(PATHS["hinchada"], "Asistencia", "asistencia_clubes_argentina.csv"))
df_asis = aplicar_mapeo(df_asis, ['Club'])

def clean_attendance(val):
    s = str(val).strip().replace('.', '')
    return int(float(s)) if s not in ['nan', '0', ''] else 0

df_asis['espectadores'] = df_asis['espectadores'].apply(clean_attendance)
df_asis.to_csv(os.path.join(PATHS["hinchada"], "Asistencia", "asistencia_estandarizada.csv"), index=False, encoding='utf-8-sig')

# 2. Unificación de Encuestas y Censo
tyc = pd.read_csv(os.path.join(PATHS["hinchada"], "Popularidad", "Encuestas", "tyc_sports_censo_crudo.csv"))
cons = pd.read_csv(os.path.join(PATHS["hinchada"], "Popularidad", "Encuestas", "consultoras_historico_crudo.csv"))
redes = pd.read_csv(os.path.join(PATHS["hinchada"], "Popularidad", "Encuestas", "redes_sociales_crudo.csv"))

dfs_pop = [aplicar_mapeo(d, ['Club']) for d in [tyc, cons, redes]]
df_pop_unificada = reduce(lambda left, right: pd.merge(left, right, on='Club', how='outer'), dfs_pop).fillna(0)

df_pop_unificada.to_csv(os.path.join(PATHS["hinchada"], "Popularidad", "popularidad_unificada_estandarizada.csv"), index=False, encoding='utf-8-sig')

print("✅ Datos de hinchada unificados y estandarizados.")

✅ Datos de hinchada unificados y estandarizados.


In [4]:
# ------------------------------------------------------------------------------
# FASE 3: Carga de datasets de infraestructura
# ------------------------------------------------------------------------------

df_infra = pd.read_csv(os.path.join(PATHS["estadios"], "estadios_infraestructura_master.csv"))
df_m2 = pd.read_csv(os.path.join(PATHS["estadios"], "valor_m2_estadios.csv"))

# Limpieza de capacidad y mapeo
df_infra = aplicar_mapeo(df_infra, ['Club'])
df_m2 = aplicar_mapeo(df_m2, ['club']).rename(columns={'club': 'Club'})

def parse_capacidad(val):
    clean = re.sub(r'[^0-9]', '', str(val))
    return int(clean) if clean else 0

df_infra['Capacidad_Wiki'] = df_infra['Capacidad_Wiki'].apply(parse_capacidad)

# Merge Final de Estadios
df_estadios_final = pd.merge(df_infra, df_m2[['Club', 'estadio', 'valor_m2_usd']], on='Club', how='outer')
df_estadios_final.to_csv(os.path.join(PATHS["estadios"], "estadios_estandarizado.csv"), index=False, encoding='utf-8-sig')

print("✅ Infraestructura de estadios estandarizada.")

✅ Infraestructura de estadios estandarizada.


In [5]:
# ------------------------------------------------------------------------------
# FASE 4: ESTANDARIZACIÓN FINANCIERA (USD Millones)
# ------------------------------------------------------------------------------

# 1. Definición de ruta basada en el archivo generado en el Módulo 1
file_input_finanzas = os.path.join(PATHS["finanzas"], "finanzas_y_valores_mercado.csv")
file_output_finanzas = os.path.join(PATHS["finanzas"], "finanzas_y_valores_mercado_estandarizado.csv")

if os.path.exists(file_input_finanzas):
    print(f"💵 Procesando: {os.path.basename(file_input_finanzas)}")
    
    # 2. Carga y Estandarización
    df_market = pd.read_csv(file_input_finanzas)
    df_market = aplicar_mapeo(df_market, ['Club'])

    # 3. Limpieza de métricas financieras
    # Nos aseguramos de que los valores sean floats para el modelo
    cols_financieras = ['Ingresos_Anuales_USD_M', 'Valor_Plantel_USD_M', 'Eficiencia_Gasto']
    for col in cols_financieras:
        if col in df_market.columns:
            df_market[col] = pd.to_numeric(df_market[col], errors='coerce').fillna(0)

    # 4. Exportación
    df_market.to_csv(file_output_finanzas, index=False, encoding='utf-8-sig')
    
    print(f"✅ Dataset financiero estandarizado correctamente.")
    print(f"📁 Guardado en: {file_output_finanzas}")
    display(df_market.head())
else:
    print(f"❌ Error crítico: No se encontró el archivo en {file_input_finanzas}")
    print("💡 Verificá que el Módulo 1 (Scraping) haya generado el CSV de finanzas.")

💵 Procesando: finanzas_y_valores_mercado.csv
✅ Dataset financiero estandarizado correctamente.
📁 Guardado en: C:\Users\Free2\Desktop\Proyecto_Localia_Rendimiento\Datos\4_Finanzas\finanzas_y_valores_mercado_estandarizado.csv


,Club,Ingresos_Anuales_USD_M,Valor_Plantel_USD_M,Eficiencia_Gasto
0,Boca Juniors,110.5,85.4,0.772851
1,River Plate,115.2,92.1,0.799479
2,Racing Club,45.8,40.2,0.877729
3,Independiente,38.0,32.5,0.855263
4,San Lorenzo de Almagro,35.5,30.1,0.847887
